<a href="https://colab.research.google.com/github/mdehghani86/AppliedGenAI/blob/main/M11_Lab2_Multi_Agent_Investment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💹 <span style="color:#2c3e50;">Advanced CrewAI:</span> <span style="color:#16a085;">Multi-Agent Investment Analysis</span> with Delegation & RAG

## 📘 <span style="color:#34495e;">Lab Overview</span>

You already know the basics of CrewAI — now let’s explore its **advanced features** that unlock real-world power.  
In this lab, you'll build a **sophisticated investment analysis system** demonstrating:

### 🎯 <span style="color:#2980b9;">Advanced Features You'll Master</span>
- 🧠 <strong>Agent Delegation</strong> – Let agents automatically assign work to specialists  
- 📄 <strong>RAG Integration</strong> – Analyze uploaded financial documents with AI  
- 📈 <strong>Real-time Data</strong> – Combine live market data with AI analysis  
- 📝 <strong>Professional Output</strong> – Transform messy AI responses into clean reports

### 💼 <span style="color:#8e44ad;">What You're Building</span>
A **4-agent investment team** that works like a real Wall Street firm:  
- The **Portfolio Manager** delegates to specialists  
- The **Research Analyst** reads your uploaded documents  
- The team produces **investment recommendations** using live data

### 🔥 <span style="color:#c0392b;">Why This Matters</span>
These patterns — delegation, RAG, real-time integration — are **essential for building production-grade AI systems** that can handle complex, multi-step workflows across any domain.


In [1]:
# +++++ 📦 Package Installation
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install required packages for advanced CrewAI features and financial data integration

!pip install -q "crewai<=1.4.0" "crewai-tools<=1.4.0" langchain-openai yfinance plotly "qdrant-client<=1.15.0"
print("✅ All packages installed successfully!")

# 📌 Package Explanations:
# - crewai: Core library to define and manage multi-agent AI workflows.
# - crewai-tools: Adds tools and enhancements to improve agent capabilities in CrewAI.
# - langchain-openai: Enables integration of OpenAI LLMs with LangChain for natural language processing.
# - yfinance: Used to fetch real-time and historical financial data from Yahoo Finance.
# - plotly: Enables creation of interactive and visually appealing charts for data analysis.

# Start time tracking (put this at the beginning of your lab)
import time
from datetime import datetime
start_time = time.time()

✅ All packages installed successfully!


In [2]:
# +++++ 🎨 Pretty Print Utility
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create styled message boxes for better visual output in Colab

from IPython.display import display, HTML

def pretty_print(text, title="ℹ️ Info", theme="blue"):
    """Displays a styled message box with optional color themes: blue, red, or yellow."""

    themes = {
        "blue": {"color": "#1e4b8f", "background": "#f0f6ff"},
        "red": {"color": "#c62828", "background": "#ffebee"},
        "yellow": {"color": "#b26a00", "background": "#fff8e1"}
    }

    style = themes.get(theme.lower(), themes["blue"])
    formatted_text = text.replace('\n', '<br>')

    display(HTML(f"""
    <div style="border-left: 5px solid {style['color']}; padding: 12px 16px; background-color: {style['background']};
                border-radius: 6px; font-family: 'Segoe UI', sans-serif; line-height: 1.6; margin: 10px 0;">
        <strong style="color: {style['color']}; font-size: 16px;">{title}</strong><br>
        <span style="font-size: 14px; color: #333;">{formatted_text}</span>
    </div>
    """))

print("🎨 Pretty print utility ready!")

🎨 Pretty print utility ready!


In [3]:
# +++++ 🔑 API Setup & Authentication
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Secure API key setup from Google Colab secrets

try:
    from google.colab import userdata
    import os
    os.environ["OPENAI_API_KEY"] = userdata.get('openai.api_key')
    pretty_print("🔐 OpenAI API key successfully loaded. You're authenticated and ready to go!", "✅ API Key Setup", "blue")
except:
    pretty_print("⚠️ OpenAI API key not found. Please set it in Colab ➤ More ➤ Secrets before running the lab.", "❌ Missing API Key", "red")


## 🛠️ Advanced Tools & Agent Setup

Now we'll set up the advanced CrewAI components that make this system powerful:

**🔧 Specialized Tools:**
- **Financial Web Scrapers** - Extract live data from Yahoo Finance and MarketWatch
- **RAG File Reader** - Analyze your uploaded financial documents (PDFs, reports)
- **Web Search** - General purpose research capabilities

**👥 The 4-Agent Investment Team:**
1. **📊 Portfolio Manager** - Has delegation powers, coordinates the entire analysis
2. **📰 Market Analyst** - Scrapes financial websites for current news and sentiment
3. **📚 Research Analyst** - Uses RAG to read and analyze your uploaded documents
4. **💹 Trading Strategist** - Synthesizes everything into actionable recommendations

**🔥 Key Advanced Features:**
- **Delegation**: Portfolio Manager can automatically assign tasks to specialists
- **RAG**: Research Analyst reads YOUR uploaded files (earnings reports, SEC filings)
- **Specialization**: Each agent has specific tools and expertise areas

In [4]:
# +++++ 🛠️ Import Libraries & Initialize Tools
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Import CrewAI framework and set up specialized tools for financial analysis

from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI
# Built-in tools for web search, web scraping, and file reading
from crewai_tools import WebsiteSearchTool, ScrapeWebsiteTool, FileReadTool

# FINANCIAL DATA SCRAPING TOOLS:
yahoo_finance_scraper = ScrapeWebsiteTool(website_url='https://finance.yahoo.com')    # Live stock prices
marketwatch_scraper = ScrapeWebsiteTool(website_url='https://www.marketwatch.com')    # Market news
web_search = WebsiteSearchTool()                                                       # General web search

# RAG (Retrieval-Augmented Generation) TOOL:
file_reader = FileReadTool()  # Reads PDFs, text files, and documents you upload

# AI MODEL CONFIGURATION:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # Lower temp = more consistent output

pretty_print("Investment tools locked and loaded!", "🔧 Tools Ready", "blue")


In [5]:
# +++++ 👥 Create Specialized AI Agents with Delegation
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create 4 specialized AI agents that work together like a real investment team

# AGENT 1: PORTFOLIO MANAGER (Simplified)
portfolio_manager = Agent(
    role="Portfolio Manager",
    goal="Coordinate team and make final investment decision",
    backstory="Wall Street portfolio manager who delegates tasks efficiently.",
    llm=llm,
    tools=[web_search],
    allow_delegation=True,
    max_delegation=2,  # Reduced from 3
    verbose=False
)

# AGENT 2: MARKET ANALYST (Simplified)
market_analyst = Agent(
    role="Market Analyst",
    goal="Find current news and sentiment",
    backstory="Financial journalist specializing in market news.",
    llm=llm,
    tools=[web_search],  # Removed marketwatch
    verbose=False
)

# AGENT 3: RESEARCH ANALYST (Simplified)
research_analyst = Agent(
    role="Research Analyst",
    goal="Analyze financial fundamentals",
    backstory="CFA analyst who reviews financial metrics.",
    llm=llm,
    tools=[web_search],
    verbose=False
)

# AGENT 4: TRADING STRATEGIST (Simplified with Risk Focus)
trading_strategist = Agent(
    role="Risk-Aware Strategist",
    goal="Create risk-adjusted trading strategy with metrics",
    backstory="""Quantitative analyst who calculates:
    - Risk/Reward = (Target-Entry)/(Entry-Stop)
    - Position Size using 2% risk rule
    Always include these metrics.""",
    llm=llm,
    tools=[web_search],
    verbose=False
)

pretty_print("💼 Investment dream team assembled!\n📊 Portfolio Manager (Boss)\n📰 Market Analyst\n📚 Research Analyst\n💹 Trading Strategist", "👥 Team Ready", "blue")


## 📋 Task Definition & Workflow Design

Here's where the advanced CrewAI features really shine. We'll create tasks that demonstrate:

**🎯 Delegation in Action:**
The Portfolio Manager doesn't do the work directly - instead, it **delegates** specific tasks to the right specialists and then **coordinates** their findings into a final recommendation.

**📚 RAG Implementation:**
The Research Analyst can read and analyze any financial documents you upload (earnings reports, SEC filings, research papers) and extract key insights that wouldn't be available through web search alone.

**🔄 Workflow Process:**
1. Portfolio Manager **delegates** news analysis to Market Analyst
2. Portfolio Manager **delegates** document analysis to Research Analyst  
3. Portfolio Manager **delegates** strategy creation to Trading Strategist
4. Portfolio Manager **synthesizes** all findings into executive summary

**💡 Why This Architecture Works:**
Just like a real investment firm, specialization + coordination produces better results than any single agent trying to do everything.

In [6]:
# +++++ 📋 Define Agent Tasks with Specific Formats
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create specific tasks for each agent with exact output formats to avoid messy results

def create_enhanced_investment_tasks(stock_symbol):
    """Creates concise tasks to avoid context overflow."""

    coordination_task = Task(
        description=f"""Coordinate {stock_symbol} analysis. Delegate to team.

        Output:
        {stock_symbol}: [BUY/SELL/HOLD]
        Target: $X | Stop: $X
        Risk/Reward: X:1 | Position: X%
        Key insight: [brief]
        Risk: [brief]""",
        agent=portfolio_manager,
        expected_output=f"Brief summary for {stock_symbol}"
    )

    market_task = Task(
        description=f"""Get {stock_symbol} latest news.

        Output:
        2 headlines
        Sentiment: [Positive/Negative/Neutral]""",
        agent=market_analyst,
        expected_output=f"News for {stock_symbol}"
    )

    research_task = Task(
        description=f"""Analyze {stock_symbol} financials briefly.

        Output:
        Revenue trend
        Profit status
        Key metric""",
        agent=research_analyst,
        expected_output=f"Financials for {stock_symbol}"
    )

    strategy_task = Task(
        description=f"""Create brief strategy for {stock_symbol}.

        Output:
        Action: [BUY/SELL/HOLD]
        Entry: $X | Target: $X | Stop: $X
        Risk/Reward: [calculate]
        Position %: [calculate using 2% rule]
        Brief reason""",
        agent=trading_strategist,
        expected_output=f"Strategy for {stock_symbol}"
    )

    return coordination_task, market_task, research_task, strategy_task

print("📋 Task templates created - ready for delegation!")

📋 Task templates created - ready for delegation!


## 🚀 Crew Assembly & Advanced Output Processing

The final step brings everything together with two key advanced features:

**🎯 Multi-Agent Orchestration:**
The `analyze_stock()` function creates a crew where agents can delegate tasks to each other, work in parallel when possible, and coordinate their findings automatically.

**📊 Real-Time Data Integration:**
The `clean_investment_output()` function demonstrates how to enhance AI analysis with live data - it fetches current stock prices, recent trading history, and key metrics from financial APIs, then formats everything into a professional investment report.

**💡 Why This Approach Works:**
- **Delegation** ensures the right specialist handles each task
- **RAG** incorporates your private documents into the analysis  
- **Real-time data** keeps recommendations current and actionable
- **Clean formatting** transforms messy AI output into professional reports

This pattern can be adapted for any domain where you need specialized AI agents working with both private documents and live data sources.

In [7]:
# +++++ 🚀 Crew Assembly & Execution Function
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create main function that assembles all agents into a working crew

def analyze_stock(stock_symbol):
    """Analysis with reduced iterations to avoid token overflow."""

    coord_task, market_task, research_task, strategy_task = create_enhanced_investment_tasks(stock_symbol)

    investment_crew = Crew(
        agents=[portfolio_manager, market_analyst, research_analyst, trading_strategist],
        tasks=[coord_task, market_task, research_task, strategy_task],
        verbose=False,
        max_iter=2,  # Reduced from 5
        output_log_file=False
    )

    pretty_print(f"Analyzing {stock_symbol}...", "💹 Processing", "yellow")

    result = investment_crew.kickoff()
    return result

pretty_print("✅ Main analysis logic is ready to run!", title="🟢 CrewAI Initialized")

In [8]:
# +++++ 🎯 Execute Advanced CrewAI Analysis
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


# Execute the complete analysis for Apple stock
print("Running advanced CrewAI analysis with delegation and real-time data...")
result = analyze_stock('AAPL')
print(result)

pretty_print("✅ Analysis complete! Your AI investment team used delegation to coordinate specialists, integrated real-time market data, and produced a professional investment report.", "🎊 Success", "blue")

Running advanced CrewAI analysis with delegation and real-time data...


Action: BUY
Entry: $259.37 | Target: $295.07 | Stop: $245.00
Risk/Reward: (295.07 - 259.37) / (259.37 - 245.00) = 35.70 / 14.37 = 2.48:1
Position %: 2% of account / (Entry - Stop) = 0.02 / (259.37 - 245.00) = 0.02 / 14.37 = 0.00139 or approximately 0.139% of the account
Brief reason: AAPL has shown strong financial performance with record quarterly revenue and positive market sentiment, despite potential risks from market volatility and regulatory challenges.


In [9]:
# +++++ 📊 HTML Investment Report (Dropbox Version with CrewAI Data)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Download HTML report from Dropbox and use it with real CrewAI result data

import requests

print("🔄 Loading HTML investment report from Dropbox...")

# Download the HTML report script
dropbox_url = "https://www.dropbox.com/scl/fi/kwuifm4ullucyvyxnnax0/HTML-Investment-Report.html?rlkey=jkfjmxnoks7rxwuxfvh46slap&dl=1"

response = requests.get(dropbox_url)
code_content = response.text

# Execute the code to load the function
display(HTML(code_content))

# print(f"🎨 Generating HTML report with CrewAI result data...")

# # Pass the result from your earlier CrewAI analysis
# html_report = create_html_investment_report(symbol, result)

# display(HTML(html_report))

# print("✅ HTML Investment Report with real AI recommendations displayed!")

🔄 Loading HTML investment report from Dropbox...


╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 559db802-5295-4be7-a7f8-88fc14490552                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/559db802-5295-4be7-a7f8-88fc14490552?access_code=TRA │
│ CE-e92c46ebd6                                                                                                   │
│ 🔑 Access Code: TRACE-e92c46ebd6                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 🧪 Hands-On Lab: Customize and Explore Your Investment Crew

In this lab, you'll modify your CrewAI investment model by experimenting with agent roles, adding financial tools, and testing different stock symbols. Follow the steps below and submit your observations.

---

### <span style="color:#3b82f6; font-weight:bold;">1. Modify the Strategic Agent</span>
Update the Strategic Agent’s role, goal, or tools to see how it affects the model.
- You might make it more risk-focused or give it access to tools like news search or valuation metrics.

---

### <span style="color:#3b82f6; font-weight:bold;">2. Add a Financial Tool or Calculator</span>
Create a simple helper function or add a tool to compute key financial metrics.
- For example: risk/reward ratio, moving averages, or P/E ratio.

---

### <span style="color:#3b82f6; font-weight:bold;">3. Test New Stock Symbols</span>
Run the model using at least three other stock symbols:
- Suggestions: `TSLA`, `GOOGL`, and `NVDA`. Compare how the recommendations change across different companies.

---

### <span style="color:#3b82f6; font-weight:bold;">4. Submit a 1-Page PDF Report</span>
Write a brief summary covering:What you changed, What symbols you tested, What you observed

Export the report as a **1-2 page PDF**.

---

### <span style="color:#3b82f6; font-weight:bold;">5. Confirm Your Submission</span>
Complete the next cell to finalize your lab submission.
- Make sure your code is saved and your PDF is ready.

---


In [23]:
!pip install crewai crewai-tools langchain-openai yfinance beautifulsoup4 requests lxml -q

In [24]:
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI
from crewai.tools import tool
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import json

In [25]:
from crewai.tools import tool

In [26]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

In [27]:
@tool("Stock Web Scraper")
def stock_web_scraper(symbol: str) -> str:
    """
    Scrape basic stock-related information from Yahoo Finance.
    Input should be a stock ticker symbol like AAPL, TSLA, GOOGL, or NVDA.
    """
    try:
        url = f"https://finance.yahoo.com/quote/{symbol}"
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        page_text = soup.get_text(" ", strip=True)

        return page_text[:3000]
    except Exception as e:
        return f"Error scraping stock data for {symbol}: {str(e)}"

In [28]:
def risk_reward_calculator(entry, target, stop, portfolio=10000, risk_percent=2):
    """
    Calculate risk/reward ratio and suggested position size.
    """
    try:
        risk_per_share = entry - stop
        reward_per_share = target - entry

        if risk_per_share <= 0:
            return {
                "error": "Invalid setup: stop price must be below entry price."
            }

        rr_ratio = round(reward_per_share / risk_per_share, 2)
        capital_at_risk = portfolio * (risk_percent / 100)
        position_size = int(capital_at_risk / risk_per_share)

        return {
            "entry": entry,
            "target": target,
            "stop": stop,
            "risk_reward_ratio": rr_ratio,
            "capital_at_risk": round(capital_at_risk, 2),
            "position_size": position_size
        }
    except Exception as e:
        return {"error": str(e)}

In [29]:
sample_calc = risk_reward_calculator(entry=100, target=120, stop=95)
print("Sample Risk/Reward Calculation:")
print(sample_calc)

Sample Risk/Reward Calculation:
{'entry': 100, 'target': 120, 'stop': 95, 'risk_reward_ratio': 4.0, 'capital_at_risk': 200.0, 'position_size': 40}


In [30]:
def get_stock_metrics(symbol):
    try:
        stock = yf.Ticker(symbol)
        info = stock.info

        metrics = {
            "symbol": symbol,
            "current_price": info.get("currentPrice"),
            "market_cap": info.get("marketCap"),
            "trailing_pe": info.get("trailingPE"),
            "fifty_two_week_high": info.get("fiftyTwoWeekHigh"),
            "fifty_two_week_low": info.get("fiftyTwoWeekLow"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "beta": info.get("beta"),
            "long_name": info.get("longName")
        }
        return metrics
    except Exception as e:
        return {"symbol": symbol, "error": str(e)}

In [31]:
market_analyst = Agent(
    role="Market Research Analyst",
    goal="Analyze market trend and positioning of the stock using scraped stock information.",
    backstory="You analyze stock direction, market positioning, and broad trend signals.",
    llm=llm,
    tools=[stock_web_scraper],
    verbose=True,
    max_iter=1,
    allow_delegation=False
)

research_analyst = Agent(
    role="Research Analyst",
    goal="Analyze stock fundamentals and business risks using scraped data and summary metrics.",
    backstory="You focus on fundamentals, valuation, growth, and business risk.",
    llm=llm,
    tools=[stock_web_scraper],
    verbose=True,
    max_iter=1,
    allow_delegation=False
)

trading_strategist = Agent(
    role="Risk-Aware Strategist",
    goal="Create a conservative investment strategy using stock metrics and risk-aware thinking.",
    backstory="You are a disciplined strategist focused on capital preservation, practical entry-target-stop planning, and structured recommendations.",
    llm=llm,
    verbose=True,
    max_iter=1,
    allow_delegation=False
)

portfolio_manager = Agent(
    role="Portfolio Manager",
    goal="Combine all analysis and provide a final BUY, HOLD, or SELL recommendation.",
    backstory="You synthesize all previous outputs into one final decision.",
    llm=llm,
    verbose=True,
    max_iter=1,
    allow_delegation=False
)

In [32]:
def create_investment_tasks(symbol, metrics, calc_result):
    market_task = Task(
        description=f"""
        Analyze the market trend and positioning for {symbol}.

        Use the Stock Web Scraper tool once.
        Keep the answer short and practical in 3-4 lines.
        Focus on:
        - trend
        - positioning
        - overall market view
        """,
        agent=market_analyst,
        expected_output=f"Short market trend summary for {symbol}"
    )

    research_task = Task(
        description=f"""
        Analyze the fundamentals and risks for {symbol}.

        Here are some stock metrics:
        {json.dumps(metrics, indent=2)}

        Use the Stock Web Scraper tool once.
        Keep the answer short and practical in 3-4 lines.
        Focus on:
        - business quality
        - valuation
        - growth
        - risks
        """,
        agent=research_analyst,
        expected_output=f"Short fundamental analysis for {symbol}"
    )

    strategy_task = Task(
        description=f"""
        Create a short risk-aware strategy for {symbol}.

        Use these stock metrics:
        {json.dumps(metrics, indent=2)}

        Use this precomputed risk/reward analysis:
        {json.dumps(calc_result, indent=2)}

        Output should include:
        - Action: BUY / HOLD / SELL
        - Entry
        - Target
        - Stop Loss
        - Risk/Reward view
        - One-line reason

        Keep it concise.
        """,
        agent=trading_strategist,
        expected_output=f"Short risk-aware strategy for {symbol}"
    )

    final_task = Task(
        description=f"""
        Based on the market analysis, fundamental analysis, and strategy for {symbol},
        provide a final recommendation.

        Output:
        - Final Recommendation: BUY / HOLD / SELL
        - Confidence: Low / Medium / High
        - 2-3 short reasons

        Keep it brief and decision-oriented.
        """,
        agent=portfolio_manager,
        context=[market_task, research_task, strategy_task],
        expected_output=f"Final recommendation for {symbol}"
    )

    return [market_task, research_task, strategy_task, final_task]

In [33]:
def analyze_stock(symbol):
    metrics = get_stock_metrics(symbol)

    if metrics.get("current_price") is not None:
        entry = round(metrics["current_price"], 2)
        target = round(entry * 1.10, 2)
        stop = round(entry * 0.95, 2)
    else:
        entry = 100
        target = 110
        stop = 95

    calc_result = risk_reward_calculator(
        entry=entry,
        target=target,
        stop=stop,
        portfolio=10000,
        risk_percent=2
    )

    print(f"\nLive metrics for {symbol}:")
    print(json.dumps(metrics, indent=2))

    print(f"\nRisk/Reward calculation for {symbol}:")
    print(json.dumps(calc_result, indent=2))

    tasks = create_investment_tasks(symbol, metrics, calc_result)

    crew = Crew(
        agents=[
            market_analyst,
            research_analyst,
            trading_strategist,
            portfolio_manager
        ],
        tasks=tasks,
        process=Process.sequential,
        verbose=True
    )

    result = crew.kickoff()
    return result

In [34]:
result_tsla = analyze_stock("TSLA")
print("\nFinal Output for TSLA:\n")
print(result_tsla)


Live metrics for TSLA:
{
  "symbol": "TSLA",
  "current_price": 350.2499,
  "market_cap": 1314288959488,
  "trailing_pe": 324.30545,
  "fifty_two_week_high": 498.83,
  "fifty_two_week_low": 214.25,
  "sector": "Consumer Cyclical",
  "industry": "Auto Manufacturers",
  "beta": 1.915,
  "long_name": "Tesla, Inc."
}

Risk/Reward calculation for TSLA:
{
  "entry": 350.25,
  "target": 385.28,
  "stop": 332.74,
  "risk_reward_ratio": 2.0,
  "capital_at_risk": 200.0,
  "position_size": 11
}


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e935c21f-2a4e-4888-a436-931cb9cf5198                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the market trend and positioning for TSLA.                                                     │
│                                                                                                                 │
│          Use the Stock Web Scraper tool once.                                                                   │
│          Keep the answer short and practical in 3-4 lines.                                                      │
│          Focus on:                                                                                              │
│          - trend                                                                                                │
│          - positioning                                                                                          │
│          - overall market view                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Portfolio Manager                                                                                       │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Based on the market analysis, fundamental analysis, and strategy for TSLA,                             │
│          provide a final recommendation.                                                                        │
│                                                                                                                 │
│          Output:                                                                                                │
│          - Final Recommendation: BUY / HOLD / SELL                                                              │
│          - Confidence: Low / Medium / High                                                                      │
│          - 2-3 short reasons                                                                                    │
│                                                                                                                 │
│          Keep it brief and decision-oriented.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: d0eda44a-41ec-4eb8-9ff6-4fc929c2ab94                                                                     │
│  Agent: Risk-Aware Strategist                                                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Portfolio Manager                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  BUY                                                                                                            │
│  Confidence: Medium                                                                                             │
│  Reasons:                                                                                                       │
│  1. Tesla's strong market position in the electric vehicle sector suggests significant growth potential         │
│  despite current volatility.                                                                                    │
│  2. The favorable risk/reward ratio of 2.0 indicates a reasonable opportunity for profit, with a clear target   │
│  and stop-loss strategy in place.                                                                               │
│  3. While the stock is currently overvalued, the optimism surrounding Tesla's innovation and market leadership  │
│  may drive future price increases.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e935c21f-2a4e-4888-a436-931cb9cf5198                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: BUY                                                                                              │
│  Confidence: Medium                                                                                             │
│  Reasons:                                                                                                       │
│  1. Tesla's strong market position in the electric vehicle sector suggests significant growth potential         │
│  despite current volatility.                                                                                    │
│  2. The favorable risk/reward ratio of 2.0 indicates a reasonable opportunity for profit, with a clear target   │
│  and stop-loss strategy in place.                                                                               │
│  3. While the stock is currently overvalued, the optimism surrounding Tesla's innovation and market leadership  │
│  may drive future price increases.                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [35]:
test_symbols = ["TSLA", "GOOGL", "NVDA"]
results = {}

for symbol in test_symbols:
    print("\n" + "=" * 80)
    print(f"Running analysis for {symbol}")
    print("=" * 80)

    try:
        result = analyze_stock(symbol)
        results[symbol] = result
        print("\nFinal Output:\n")
        print(result)
    except Exception as e:
        results[symbol] = f"Error: {str(e)}"
        print(f"Error while analyzing {symbol}: {e}")


Running analysis for TSLA

Live metrics for TSLA:
{
  "symbol": "TSLA",
  "current_price": 350.015,
  "market_cap": 1313613807616,
  "trailing_pe": 324.1389,
  "fifty_two_week_high": 498.83,
  "fifty_two_week_low": 214.25,
  "sector": "Consumer Cyclical",
  "industry": "Auto Manufacturers",
  "beta": 1.915,
  "long_name": "Tesla, Inc."
}

Risk/Reward calculation for TSLA:
{
  "entry": 350.01,
  "target": 385.01,
  "stop": 332.51,
  "risk_reward_ratio": 2.0,
  "capital_at_risk": 200.0,
  "position_size": 11
}


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6020ae14-65ad-4c61-9b58-16ae31fcbf31                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the market trend and positioning for TSLA.                                                     │
│                                                                                                                 │
│          Use the Stock Web Scraper tool once.                                                                   │
│          Keep the answer short and practical in 3-4 lines.                                                      │
│          Focus on:                                                                                              │
│          - trend                                                                                                │
│          - positioning                                                                                          │
│          - overall market view                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Maximum iterations reached. Requesting final answer.

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Portfolio Manager                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  BUY                                                                                                            │
│  - Confidence: High                                                                                             │
│  - 2-3 short reasons:                                                                                           │
│  1. NVIDIA's leadership in AI and graphics processing positions it for sustained growth, supported by strong    │
│  demand for its products.                                                                                       │
│  2. The stock's fundamentals indicate robust earnings growth, justifying its current valuation despite the      │
│  high P/E ratio.                                                                                                │
│  3. The favorable risk/reward ratio of 2.0, with a clear entry and stop-loss strategy, offers a calculated      │
│  opportunity for investors.                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: c8acf582-cfeb-4c2c-ae33-5b6b64f7d2f2                                                                     │
│  Agent: Portfolio Manager                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Output:

BUY  
- Confidence: High  
- 2-3 short reasons:  
1. NVIDIA's leadership in AI and graphics processing positions it for sustained growth, supported by strong demand for its products.  
2. The stock's fundamentals indicate robust earnings growth, justifying its current valuation despite the high P/E ratio.  
3. The favorable risk/reward ratio of 2.0, with a clear entry and stop-loss strategy, offers a calculated opportunity for investors.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ba524166-b8ca-4a13-8ec2-7902117e8d39                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: BUY                                                                                              │
│  - Confidence: High                                                                                             │
│  - 2-3 short reasons:                                                                                           │
│  1. NVIDIA's leadership in AI and graphics processing positions it for sustained growth, supported by strong    │
│  demand for its products.                                                                                       │
│  2. The stock's fundamentals indicate robust earnings growth, justifying its current valuation despite the      │
│  high P/E ratio.                                                                                                │
│  3. The favorable risk/reward ratio of 2.0, with a clear entry and stop-loss strategy, offers a calculated      │
│  opportunity for investors.                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [36]:
for symbol, output in results.items():
    print("\n" + "-" * 80)
    print(f"{symbol} SUMMARY")
    print("-" * 80)
    print(output)


--------------------------------------------------------------------------------
TSLA SUMMARY
--------------------------------------------------------------------------------
BUY  
Confidence: Medium  
Reasons: 1. Tesla is experiencing a bullish trend driven by strong demand for electric vehicles and technological advancements, positioning it well for future growth. 2. Despite a high P/E ratio indicating potential overvaluation, the strong brand and market optimism provide a favorable risk/reward ratio for entry. 3. The stock's volatility suggests a strategic entry point with defined stop-loss and target levels, making it a calculated investment opportunity.

--------------------------------------------------------------------------------
GOOGL SUMMARY
--------------------------------------------------------------------------------
BUY  
Confidence: High  
Reasons: 1. GOOGL's strong fundamentals and solid market positioning in digital advertising and cloud services indicate robust gro

In [37]:
# +++++ 🎓 Lab Completion Certificate (Dropbox Version)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Download and run completion certificate from Dropbox

import requests

print("🔄 Loading completion certificate from Dropbox...")

# Download and execute the completion script
dropbox_url = "https://www.dropbox.com/scl/fi/5molmat6myeqaf96kp50v/CrewAI_Completiton.py?rlkey=7v7yaf9gi5hupkxiqaits50rd&dl=1"

response = requests.get(dropbox_url)
exec(response.text)

🔄 Loading completion certificate from Dropbox...
Using existing start time from previous cell...
Congratulations on completing the Advanced CrewAI Lab!
Please enter your information for certification:
--------------------------------------------------
Enter your full name: Simran Abhay Sinha
Enter your student ID: 002475433
How long did it take you to complete this lab? (in minutes): 300

ACADEMIC INTEGRITY CONFIRMATION
----------------------------------------
Do you confirm you have completed this lab all by yourself and understand the concepts? (yes/no): Yes

SKILLS REFLECTION
Please answer these questions (4-5 words each):
---------------------------------------------
How does agent delegation work? Agents assign tasks to others
What is RAG integration? Retrieval augmented generation with documents
Why use real-time data? Provides current and accurate information



ðŸŽ‰ Well done, Simran Abhay Sinha! You've mastered advanced AI agent collaboration!
